In [1]:
!pip install numpy pandas matplotlib


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [10]:
import pandas as pd

metrics_df: pd.DataFrame = pd.read_csv('metrics/metrics_all_ncu.csv')
metrics_df

,GpuName,SeqLen,HeadDim,Run,Range,Action,dram__cycles_active.avg,gpu__time_duration.sum,l1tex__cycles_active.avg,lts__cycles_active.avg,...,smsp__pcsamp_warps_issue_stalled_short_scoreboard,smsp__warps_issue_stalled_long_scoreboard.avg,smsp__warps_issue_stalled_math_pipe_throttle.avg,smsp__warps_issue_stalled_mio_throttle.avg,smsp__warps_issue_stalled_not_selected.avg,smsp__warps_issue_stalled_short_scoreboard.avg,smsp__warps_issue_stalled_wait.avg,sm__pipe_aluheavy_cycles_active.avg,sm__pipe_fmaheavy_cycles_active.avg,sm__pipe_fmalite_cycles_active.avg
0,a100-sxm4-40gb,256,64,0,0,unrolled_elementwise_kernel,50.7,7456.0,1.601593e+03,1.757700e+03,...,1.0,6.297593e+02,0.000000e+00,0.00000,0.000000e+00,29.888889,2.408889e+02,NaN,NaN,NaN
1,a100-sxm4-40gb,256,64,0,0,unrolled_elementwise_kernel,50.7,7424.0,1.583741e+03,1.842463e+03,...,1.0,6.119074e+02,0.000000e+00,0.00000,0.000000e+00,29.824074,2.408889e+02,NaN,NaN,NaN
2,a100-sxm4-40gb,256,64,0,0,unrolled_elementwise_kernel,50.7,7392.0,1.582528e+03,1.733312e+03,...,3.0,6.324398e+02,0.000000e+00,0.00000,0.000000e+00,29.761574,2.408889e+02,NaN,NaN,NaN
3,a100-sxm4-40gb,256,64,0,0,vectorized_elementwise_kernel,54.9,3616.0,2.510556e+02,9.283125e+02,...,0.0,1.009421e+02,0.000000e+00,0.00000,0.000000e+00,1.629630,1.111111e+01,NaN,NaN,NaN
4,a100-sxm4-40gb,256,64,0,0,vectorized_elementwise_kernel,54.9,3616.0,2.438796e+02,1.060450e+03,...,0.0,9.570602e+01,0.000000e+00,0.00000,0.000000e+00,1.629630,1.111111e+01,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2566,t4,8192,128,2,0,volta_sgemm_128x64_nn,9625981.5,5933664.0,3.467783e+06,5.058044e+06,...,544.0,7.826174e+04,3.940338e+06,338280.45625,3.955725e+06,51981.656250,1.556349e+06,NaN,NaN,NaN
2567,t4,8192,128,2,0,vectorized_elementwise_kernel,92463.0,24160.0,1.223858e+04,1.769500e+04,...,6.0,5.292419e+04,1.435188e+02,210.20625,2.052250e+02,1051.981250,3.356756e+03,NaN,NaN,NaN
2568,t4,8192,128,0,0,fmha_cutlassF_f16_aligned_32x128_rf_sm75,411159.0,5579392.0,2.897558e+06,3.540478e+06,...,18328.0,1.083766e+06,1.906061e+05,389439.61250,1.069107e+05,915427.650000,1.209729e+06,NaN,NaN,NaN
2569,t4,8192,128,1,0,fmha_cutlassF_f16_aligned_32x128_rf_sm75,412807.0,5591008.0,2.903576e+06,3.496220e+06,...,18373.0,1.072384e+06,1.908920e+05,391341.08125,1.065178e+05,917177.125000,1.210074e+06,NaN,NaN,NaN


In [4]:
metrics_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2571 entries, 0 to 2570
Data columns (total 45 columns):
 #   Column                                                         Non-Null Count  Dtype  
---  ------                                                         --------------  -----  
 0   GpuName                                                        2571 non-null   str    
 1   SeqLen                                                         2571 non-null   int64  
 2   HeadDim                                                        2571 non-null   int64  
 3   Run                                                            2571 non-null   int64  
 4   Range                                                          2571 non-null   int64  
 5   Action                                                         2571 non-null   str    
 6   dram__cycles_active.avg                                        2571 non-null   float64
 7   gpu__time_duration.sum                                         2571 non

In [5]:
sm_cycles_elapsed = metrics_df['sm__cycles_elapsed.avg']

In [19]:
util_df = metrics_df[['GpuName', 'SeqLen', 'HeadDim', 'Run', 'Action']]
util_df['SM_Util'] = metrics_df['sm__cycles_active.avg'] / sm_cycles_elapsed
util_df['SMSP_Util'] = metrics_df['smsp__issue_active.avg.pct_of_peak_sustained_elapsed']
util_df['Tensor_Util'] = metrics_df['sm__pipe_tensor_cycles_active.avg'] / sm_cycles_elapsed
util_df['FMA_Util'] = metrics_df['sm__pipe_fma_cycles_active.avg'] / sm_cycles_elapsed
util_df['FMA_Heavy_Util'] = metrics_df['sm__pipe_fmaheavy_cycles_active.avg'] / sm_cycles_elapsed
util_df['FMA_Lite_Util'] = metrics_df['sm__pipe_fmalite_cycles_active.avg'] / sm_cycles_elapsed
util_df['SFU_Util'] = metrics_df['smsp__inst_executed_pipe_xu.avg.pct_of_peak_sustained_elapsed']
util_df['Shared_Util'] = metrics_df['sm__pipe_shared_cycles_active.avg'] / sm_cycles_elapsed
util_df['ALU_Util'] = metrics_df['sm__pipe_alu_cycles_active.avg'] / sm_cycles_elapsed
util_df['ALU_Heavy_Util'] = metrics_df['sm__pipe_aluheavy_cycles_active.avg'] / sm_cycles_elapsed
# util_df['ALU_Lite_Util'] = metrics_df['sm__pipe_alulite_cycles_active.avg'] / sm_cycles_elapsed
# util_df['FP16_Util'] = metrics_df['sm__pipe_fp16_cycles_active.avg'] / sm_cycles_elapsed
util_df['FP64_Util'] = metrics_df['sm__pipe_fp64_cycles_active.avg'] / sm_cycles_elapsed
util_df['L1_Cache_Util'] = metrics_df['l1tex__cycles_active.avg'] / sm_cycles_elapsed
# util_df['L1_Cache_Util_Ampere+'] = metrics_df['1tex__cycles_active.avg'] / sm_cycles_elapsed
util_df['LTS_Util'] = metrics_df['lts__cycles_active.avg'] / sm_cycles_elapsed
util_df['DRAM_Util'] = metrics_df['dram__cycles_active.avg'] / sm_cycles_elapsed
util_df

,GpuName,SeqLen,HeadDim,Run,Action,SM_Util,SMSP_Util,Tensor_Util,FMA_Util,FMA_Heavy_Util,FMA_Lite_Util,SFU_Util,Shared_Util,ALU_Util,ALU_Heavy_Util,FP64_Util,L1_Cache_Util,LTS_Util,DRAM_Util
0,a100-sxm4-40gb,256,64,0,unrolled_elementwise_kernel,0.197860,0.735747,0.000000,0.010542,NaN,NaN,0.000000,0.000000,0.019327,NaN,0.000000,0.197860,0.217145,0.006263
1,a100-sxm4-40gb,256,64,0,unrolled_elementwise_kernel,0.196005,0.737065,0.000000,0.010561,NaN,NaN,0.000000,0.000000,0.019362,NaN,0.000000,0.196005,0.228025,0.006275
2,a100-sxm4-40gb,256,64,0,unrolled_elementwise_kernel,0.197519,0.743326,0.000000,0.010651,NaN,NaN,0.000000,0.000000,0.019526,NaN,0.000000,0.197519,0.216339,0.006328
3,a100-sxm4-40gb,256,64,0,vectorized_elementwise_kernel,0.064764,0.122296,0.000306,0.003975,NaN,NaN,0.000000,0.000306,0.000917,NaN,0.000000,0.064764,0.239475,0.014162
4,a100-sxm4-40gb,256,64,0,vectorized_elementwise_kernel,0.062554,0.121597,0.000304,0.003952,NaN,NaN,0.000000,0.000304,0.000912,NaN,0.000000,0.062554,0.271999,0.014082
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2566,t4,8192,128,2,volta_sgemm_128x64_nn,0.999017,56.454029,0.000000,3.875189,NaN,NaN,0.000000,0.000000,0.188409,NaN,0.000000,0.999017,1.457147,2.773101
2567,t4,8192,128,2,vectorized_elementwise_kernel,0.869674,5.601932,0.000000,0.116425,NaN,NaN,11.642483,0.000000,0.087319,NaN,0.000000,0.869674,1.257408,6.570428
2568,t4,8192,128,0,fmha_cutlassF_f16_aligned_32x128_rf_sm75,0.887759,22.171473,1.043686,0.366022,NaN,NaN,6.505734,1.043686,0.625897,NaN,0.000502,0.887759,1.084737,0.125972
2569,t4,8192,128,1,fmha_cutlassF_f16_aligned_32x128_rf_sm75,0.887750,22.125302,1.041532,0.365259,NaN,NaN,6.492186,1.041532,0.624594,NaN,0.000501,0.887750,1.068947,0.126213


In [20]:
util_df.to_csv('metrics/util_ncu.csv')